# 新疆多分类模型评估主流程
本Notebook实现了数据加载、模型训练、评估与多次统计的自动化流程，结构清晰，便于复用和维护。

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras import regularizers
import random
import time

d:\ProgramData\Anaconda3\envs\tensorflow210\lib\site-packages\tensorflow\python\framework\dtypes.py:516: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
d:\ProgramData\Anaconda3\envs\tensorflow210\lib\site-packages\tensorflow\python\framework\dtypes.py:517: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
d:\ProgramData\Anaconda3\envs\tensorflow210\lib\site-packages\tensorflow\python\framework\dtypes.py:518: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
d:\ProgramData\Anaconda3\envs\tensorflow210\lib\s

#### 工具函数

In [2]:
def set_seed(seed=42):
    """全局随机种子，保证实验可复现"""
    tf.keras.backend.clear_session()
    random.seed(seed)
    np.random.seed(seed)
    tf.set_random_seed(seed)

def setup_gpu():
    """设置GPU显存按需分配"""
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print(f"检测到{len(gpus)}块GPU，已设置显存按需分配。")
        except RuntimeError as e:
            print(e)
    else:
        print("未检测到GPU，使用CPU运行。")

#### 数据处理

In [3]:
def load_data(train_path):
    """加载训练数据，返回DataFrame"""
    data = pd.read_csv(train_path, header=0, encoding='utf_8')
    print('类别分布:')
    print(data['className'].value_counts())
    return data

def split_data(data, test_ratio=0.2):
    """打乱并划分训练/测试集"""
    idx = np.random.permutation(len(data))
    data_all = data.iloc[idx, :]
    split_idx = int(len(data_all) * (1 - test_ratio))
    train = data_all.iloc[:split_idx]
    test = data_all.iloc[split_idx:]
    return train, test

def get_xy(df):
    """提取特征和标签"""
    X = df.iloc[:, 6:]
    y = df['num'].values
    return X, y

def get_class_map(data):
    """获取类别编号到名称的映射"""
    return data.drop_duplicates(subset='num').set_index('num')['className'].to_dict()

#### 模型构建

In [4]:
def build_model(input_dim, num_classes):
    """构建全连接神经网络模型"""
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(512, activation='relu', input_dim=input_dim, kernel_regularizer=regularizers.l2(0.001)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    return model

def multi_category_focal_loss2(gamma=2., alpha=.25, class_weights=None):
    """多类别focal loss，支持类别权重"""
    epsilon = 1.e-7
    gamma = float(gamma)
    alpha = tf.constant(alpha, dtype=tf.float32)
    if class_weights is not None:
        weights = np.array([class_weights.get(i, 1.0) for i in range(len(class_weights))])
        class_weights_tf = tf.constant(weights, dtype=tf.float32)
    else:
        class_weights_tf = None
    def focal_loss_fixed(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
        alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
        y_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        ce = -tf.math.log(y_t)
        weight = tf.pow(1. - y_t, gamma)
        fl = alpha_t * weight * ce
        if class_weights_tf is not None:
            fl = fl * class_weights_tf
        return tf.reduce_mean(fl)
    return focal_loss_fixed

#### 单次训练与评估

In [5]:
def train_and_evaluate(train_df, test_df, class_map, verbose=0):
    """单次训练与评估，返回整体四项指标和各类别准确率"""
    X_train, y_train = get_xy(train_df)
    X_test, y_test = get_xy(test_df)
    num_classes = len(class_map)
    y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes=num_classes)
    # 计算类别权重
    class_weights_array = compute_class_weight(class_weight='balanced', classes=np.arange(num_classes), y=y_train)
    class_weights = dict(enumerate(class_weights_array))
    # 构建模型
    model = build_model(X_train.shape[1], num_classes)
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
    model.compile(loss=multi_category_focal_loss2(alpha=0.25, gamma=2, class_weights=class_weights),
                  optimizer=optimizer, metrics=['accuracy'])
    # 训练
    early_stopping = EarlyStopping(monitor='loss', patience=30, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='loss', factor=0.5, patience=10, verbose=0)
    model.fit(X_train, y_train_cat, epochs=500, batch_size=64, verbose=verbose, callbacks=[reduce_lr, early_stopping])
    # 预测
    y_pred = model.predict(X_test).argmax(axis=1)
    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
    cm = confusion_matrix(y_test, y_pred, labels=list(class_map.keys()))
    per_class_acc = cm.diagonal() / cm.sum(axis=1)
    return [acc, precision, recall, f1], per_class_acc

#### 多次统计主流程 

In [6]:
def multi_run_eval(data, n_runs=10, test_ratio=0.2, seed=42):
    """多次随机划分训练/测试集，训练评估并统计结果"""
    set_seed(seed)
    class_map = get_class_map(data)
    all_class_ids = sorted(class_map.keys())
    class_labels = [class_map[i] for i in all_class_ids]
    overall_scores = []
    per_class_accs = []
    for run in range(n_runs):
        print(f"\n===== 正在进行第 {run+1}/{n_runs} 次训练与评估 =====")
        train_df, test_df = split_data(data, test_ratio)
        scores, per_class_acc = train_and_evaluate(train_df, test_df, class_map, verbose=0)
        # 补齐类别准确率
        if len(per_class_acc) < len(all_class_ids):
            acc_full = np.full(len(all_class_ids), np.nan)
            acc_full[:len(per_class_acc)] = per_class_acc
            per_class_accs.append(acc_full)
        else:
            per_class_accs.append(per_class_acc)
        overall_scores.append(scores)
        print(f"Run {run+1}: acc={scores[0]:.4f}, precision={scores[1]:.4f}, recall={scores[2]:.4f}, f1={scores[3]:.4f}")
    # 汇总表格
    score_names = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
    index = score_names + class_labels
    results = np.vstack([np.array(overall_scores).T, np.array(per_class_accs).T])
    df_results = pd.DataFrame(results, index=index, columns=[f'Run_{i+1}' for i in range(n_runs)])
    df_results['Mean'] = df_results.mean(axis=1)
    print('\n多次整体与各类别准确率统计表：')
    print(df_results)
    return df_results

#### 主程序入口

In [7]:
if __name__ == '__main__':
    setup_gpu()
    data = load_data('F:/TensorFlow/xinjiang/traindata20250626_2_code.csv')
    df_results = multi_run_eval(data, n_runs=10, test_ratio=0.2, seed=42)
    df_results.to_csv('F:/TensorFlow/xinjiang/XJ10runs20250626_final.csv', encoding='utf_8_sig')
    print('\n结果已保存至 F:/TensorFlow/xinjiang/XJ10runs20250626_final.csv')

检测到1块GPU，已设置显存按需分配。
类别分布:
农田                  279
裸地                   99
芦苇草甸                 92
蒿属荒漠                 84
柽柳属荒漠                75
针茅属荒漠草原              64
梭梭荒漠                 62
针茅属草原                60
薹草属草甸                56
紫花针茅草原               49
驼绒藜荒漠                42
猪毛菜属荒漠               38
嵩草属草甸                30
芦苇盐生草甸               30
锦鸡儿属荒漠               30
盐生假木贼荒漠              26
旱蒿荒漠                 26
假木贼属荒漠               24
线叶嵩草草甸               23
早熟禾属草甸               21
麻黄荒漠                 20
绢蒿属荒漠                20
膜果麻黄荒漠               19
高山绢蒿荒漠               19
戈壁藜荒漠                18
红砂荒漠                 18
针茅属高寒草原              17
锦鸡儿荒漠草原              11
新疆银穗草、穗状寒生羊茅荒漠草原     10
新疆银穗草草原              10
鸭茅草甸                 10
冷蒿荒漠草原                9
Name: className, dtype: int64


===== 正在进行第 1/10 次训练与评估 =====
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor

===== 正在进行第 1/10 次训练与评估 =

d:\ProgramData\Anaconda3\envs\tensorflow210\lib\site-packages\ipykernel_launcher.py:26: RuntimeWarning: invalid value encountered in true_divide


Run 3: acc=0.4409, precision=0.3445, recall=0.3494, f1=0.3302

===== 正在进行第 4/10 次训练与评估 =====


d:\ProgramData\Anaconda3\envs\tensorflow210\lib\site-packages\ipykernel_launcher.py:26: RuntimeWarning: invalid value encountered in true_divide


Run 4: acc=0.3871, precision=0.3172, recall=0.3333, f1=0.3026

===== 正在进行第 5/10 次训练与评估 =====


...

Run 6: acc=0.4480, precision=0.2639, recall=0.3219, f1=0.2725

===== 正在进行第 7/10 次训练与评估 =====
Run 7: acc=0.4158, precision=0.2955, recall=0.3259, f1=0.2773

===== 正在进行第 8/10 次训练与评估 =====
Run 7: acc=0.4158, precision=0.2955, recall=0.3259, f1=0.2773

===== 正在进行第 8/10 次训练与评估 =====


d:\ProgramData\Anaconda3\envs\tensorflow210\lib\site-packages\ipykernel_launcher.py:26: RuntimeWarning: invalid value encountered in true_divide


Run 8: acc=0.4552, precision=0.3180, recall=0.3251, f1=0.2970

===== 正在进行第 9/10 次训练与评估 =====


d:\ProgramData\Anaconda3\envs\tensorflow210\lib\site-packages\ipykernel_launcher.py:26: RuntimeWarning: invalid value encountered in true_divide


Run 9: acc=0.4552, precision=0.3114, recall=0.3136, f1=0.2700

===== 正在进行第 10/10 次训练与评估 =====
Run 10: acc=0.4444, precision=0.3745, recall=0.3797, f1=0.3277

多次整体与各类别准确率统计表：
                     Run_1     Run_2     Run_3     Run_4     Run_5     Run_6  \
Accuracy          0.408602  0.458781  0.440860  0.387097  0.415771  0.448029   
Precision         0.288331  0.353298  0.344451  0.317203  0.293403  0.263936   
Recall            0.331846  0.389470  0.349413  0.333297  0.342465  0.321891   
F1 Score          0.291216  0.331877  0.330157  0.302573  0.292821  0.272517   
针茅属荒漠草原           0.052632  0.000000  0.222222  0.187500  0.000000  0.153846   
紫花针茅草原            0.500000  0.636364  0.583333  0.333333  0.636364  0.545455   
猪毛菜属荒漠            0.181818  0.230769  0.142857  0.000000  0.000000  0.000000   
高山绢蒿荒漠            0.250000  0.333333  0.500000  1.000000  0.428571  0.500000   
蒿属荒漠              0.684211  0.500000  0.750000  0.411765  0.529412  0.562500   
新疆银穗草草原           0.333333